# Prueba de estilo — ¿el tono separa una hoja sana de una con roya?

Objetivo: ver, con números inventados, si el color de la hoja alcanza para distinguir una sana de una con roya, sin mezclar la respuesta con la medición.

Este cuaderno no usa BRACOL ni RoCoLe. Sirve para mostrar cómo queda una sección cuando cada celda hace una sola cosa y se puede leer de arriba abajo.


## Armar una mesa pequeña de hojas

Todavía no tenemos las fotos reales. Armamos 80 hojas de mentira: 40 sanas y 40 con roya. A cada una le anotamos un código, el nombre de la enfermedad y tres números de color (matiz, saturación y brillo).

El matiz es la parte del color que dice si la hoja tira a verde o a amarillo. En este ejemplo las sanas salen más verdes y las de roya más amarillas, a propósito, para que el ejercicio se pueda seguir.


In [ ]:
# Entrada
import numpy as np
import pandas as pd

seed = 42
rng = np.random.default_rng(seed)
n_healthy = 40
n_rust = 40

# Qué hacemos
healthy = pd.DataFrame(
    {
        "leaf_id": [f"S{number:03d}" for number in range(1, n_healthy + 1)],
        "disease": "sana",
        "hue": rng.normal(loc=90, scale=8, size=n_healthy),
        "saturation": rng.normal(loc=140, scale=12, size=n_healthy),
        "brightness": rng.normal(loc=160, scale=15, size=n_healthy),
    }
)
rust = pd.DataFrame(
    {
        "leaf_id": [f"R{number:03d}" for number in range(1, n_rust + 1)],
        "disease": "roya",
        "hue": rng.normal(loc=28, scale=8, size=n_rust),
        "saturation": rng.normal(loc=150, scale=12, size=n_rust),
        "brightness": rng.normal(loc=145, scale=15, size=n_rust),
    }
)
leaves = pd.concat([healthy, rust], ignore_index=True)

# Salida
leaves.head()


## Separar las palabras de los números

El código de la hoja y el nombre de la enfermedad son palabras. El matiz, la saturación y el brillo son números: eso es lo que medimos.

Los separamos para verlos distintos. La tabla `leaves` sigue completa, porque más adelante cada tono tiene que viajar con su enfermedad. Si partiéramos la tabla en dos y las reordenáramos por separado, una hoja sana podría heredar el color de otra.


In [ ]:
# Entrada
# leaves: código, enfermedad y tres números de color

# Qué hacemos
leaf_id = leaves["leaf_id"]
disease_name = leaves["disease"]
leaf_color = leaves[["hue", "saturation", "brightness"]]

# Salida
print("Código:", leaf_id.iloc[0], "| enfermedad:", disease_name.iloc[0])
print("Números del tono:", list(leaf_color.columns))
leaf_color.head()


## Mirar si el matiz cambia con la enfermedad

Queremos ver, en un histograma, si las hojas sanas y las de roya caen en zonas distintas del matiz.

Esto es solo para mirar. El corte que separa las dos no se decide aquí: si lo eligiéramos con todas las hojas, las de prueba ya habrían participado.


In [ ]:
# Entrada
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

leaf_hue = leaf_color["hue"]

# Qué hacemos
figures_dir = Path("../experiments/results")
figures_dir.mkdir(parents=True, exist_ok=True)
histogram_path = figures_dir / "00-prueba-histograma-hue.png"

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    x=leaf_hue,
    hue=disease_name,
    bins=15,
    element="step",
    stat="count",
    common_norm=False,
    ax=ax,
)
ax.set_title("Matiz de la hoja, según la enfermedad")
ax.set_xlabel("Matiz (más bajo tira a amarillo, más alto tira a verde)")
ax.set_ylabel("Cantidad de hojas")
fig.tight_layout()
fig.savefig(histogram_path, dpi=120)
plt.show()

# Salida
print(f"Figura guardada en {histogram_path.as_posix()}")


## Apartar hojas para probar, antes de calcular nada

Guardamos una parte de las hojas sin tocarlas. Con el resto vamos a aprender el corte. La semilla queda fija para que el reparto sea el mismo si alguien vuelve a correr el cuaderno.

Apartamos la fila completa: código, enfermedad y tono viajan juntos.


In [ ]:
# Entrada
from sklearn.model_selection import train_test_split

# Qué hacemos
train_leaves, test_leaves = train_test_split(
    leaves,
    test_size=0.25,
    random_state=seed,
    stratify=leaves["disease"],
)

# Salida
print(f"Para aprender: {len(train_leaves)} hojas")
print(f"Para probar: {len(test_leaves)} hojas")
test_leaves["disease"].value_counts()


## Calcular el corte solo con las hojas de aprender

En las hojas de aprender, las sanas tienen un matiz promedio y las de roya otro. El corte queda en la mitad, entre esos dos promedios.

Las hojas que apartamos no entran en esta cuenta.


In [ ]:
# Entrada
train_disease = train_leaves["disease"]
train_hue = train_leaves["hue"]

# Qué hacemos
healthy_hue_mean = train_hue[train_disease == "sana"].mean()
rust_hue_mean = train_hue[train_disease == "roya"].mean()
hue_cutoff = (healthy_hue_mean + rust_hue_mean) / 2

# Salida
print(f"Matiz promedio de las sanas (aprender): {healthy_hue_mean:.1f}")
print(f"Matiz promedio de la roya (aprender): {rust_hue_mean:.1f}")
print(f"Corte: {hue_cutoff:.1f}")


## Probar el corte en las hojas que apartamos

Si el matiz de una hoja de prueba queda del lado verde del corte, la llamamos sana. Si queda del lado amarillo, la llamamos roya.

Comparamos esa llamada con la enfermedad real de esas hojas, que no se usó para poner el corte.


In [ ]:
# Entrada
test_hue = test_leaves["hue"]
test_disease = test_leaves["disease"]

# Qué hacemos
predicted_disease = np.where(
    test_hue >= hue_cutoff,
    "sana",
    "roya",
)
n_correct = int((predicted_disease == test_disease.to_numpy()).sum())
n_test = len(test_disease)

# Salida
print(f"Coinciden {n_correct} de {n_test} hojas apartadas.")


## Conclusiones

En este ejemplo inventado el matiz sí separa las sanas de las de roya, porque los números nacieron así. El corte se calculó solo con las hojas de aprender y se contó el acierto solo con las apartadas.

Con las fotos reales el orden es el mismo, y antes de apartar hay que revisar duplicados y que la misma hoja no quede en los dos lados (`docs/datos.md`). Este archivo no responde si el color alcanza en el cultivo.
